# Демо проектных упражнений модуля 1

**Редакция 11 октября 2026 года.** Сначала построим два паспорта одного вымышленного
ассистента по документам. Для модельного паспорта используем тот же проверяемый
классификатор, что и в §17 лекции. Для системного — явно зададим источники, действия
и правило приёмки. Затем сравним два учебных среза реестра выпусков.

Это самостоятельный опыт на CPU без внешних пакетов, сети и API. `Run all` выполняет
обязательный путь целиком. Реальная LLM не вызывается; проверка готовых ответов имеет
узкий ручной эталон и не является универсальным судьёй смысла.

In [ ]:
import json


## Демо 1. Два паспорта и проверка ответа

Конфигурация описывает устройство модели. Она не сообщает, какие документы доступны
ассистенту и что считать правильным ответом. Сначала восстановим структурные поля.
Функции ниже совпадают с §17 и отклоняют невозможные размеры и противоречащие поля.
Имена конфигураций обозначают вымышленные учебные варианты; их качество неизвестно.

In [ ]:
"""Минимальный паспорт по проверяемым полям config.json."""
import json

EXAMPLE_CONFIG_JSON = r"""
{
  "hidden_size": 2880,
  "num_hidden_layers": 24,
  "num_attention_heads": 64,
  "num_key_value_heads": 8,
  "num_local_experts": 32,
  "num_experts_per_tok": 4,
  "max_position_embeddings": 131072,
  "vocab_size": 201088,
  "rope_scaling": {"rope_type": "yarn", "factor": 32}
}
"""


def positive_int(config: dict, key: str) -> int:
    """Прочитать обязательное положительное целое поле."""

    value = config.get(key)
    if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
        raise ValueError(f"{key} должно быть положительным целым")
    return value


def one_of(config: dict, *keys: str):
    """Прочитать поле-синоним и отклонить противоречие."""

    found = [(key, config[key]) for key in keys if config.get(key) is not None]
    if not found:
        return None
    if any(type(value) is not type(found[0][1]) or value != found[0][1]
           for _, value in found[1:]):
        raise ValueError(f"противоречащие поля-синонимы: {[key for key, _ in found]}")
    return found[0][1]


def attention_kind(config: dict) -> str:
    """Различить MHA, MQA, GQA и MLA по поддерживаемым полям."""

    if "kv_lora_rank" in config:  # сначала проверяем признак MLA
        return f"MLA (kv_rank={positive_int(config, 'kv_lora_rank')})"
    heads = positive_int(config, "num_attention_heads")
    kv_heads = config.get("num_key_value_heads", heads)
    if isinstance(kv_heads, bool) or not isinstance(kv_heads, int) or kv_heads <= 0:
        raise ValueError("num_key_value_heads должно быть положительным целым")
    if kv_heads > heads or heads % kv_heads:
        raise ValueError("число KV-голов должно делить число Q-голов")
    if kv_heads == heads:
        return f"MHA ({heads} голов)"
    if kv_heads == 1:
        return f"MQA ({heads} Q-голов / 1 KV)"
    return f"GQA ({heads} Q-голов / {kv_heads} KV)"


def moe_kind(config: dict) -> str:
    """Различить плотную модель и MoE с учётом полей-синонимов."""

    experts = one_of(config, "num_local_experts", "n_routed_experts", "num_experts")
    if experts is None:
        if any(config.get(k) is not None for k in ("num_experts_per_tok", "num_experts_per_token", "n_shared_experts", "num_shared_experts")):
            raise ValueError("настройки экспертов требуют их количества")
        return "dense"
    if isinstance(experts, bool) or not isinstance(experts, int) or experts <= 0:
        raise ValueError("expert count должно быть положительным целым")
    top_k = one_of(config, "num_experts_per_tok", "num_experts_per_token")
    if isinstance(top_k, bool) or not isinstance(top_k, int) or not 1 <= top_k <= experts:
        raise ValueError("top-k должно лежать в диапазоне от 1 до числа экспертов")
    shared = one_of(config, "n_shared_experts", "num_shared_experts")
    shared = 0 if shared is None else shared
    if isinstance(shared, bool) or not isinstance(shared, int) or shared < 0:
        raise ValueError("число общих экспертов должно быть неотрицательным целым")
    return f"MoE: {experts} экспертов, top-{top_k}" + (f" + {shared} общих" if shared else "")


def passport(config: dict) -> dict:
    """Вернуть структурный паспорт без сетевого вызова."""

    return {
        "layers_x_hidden": (
            positive_int(config, "num_hidden_layers"),
            positive_int(config, "hidden_size"),
        ),
        "attention": attention_kind(config),
        "ffn": moe_kind(config),
        "context": positive_int(config, "max_position_embeddings"),
        "vocabulary": positive_int(config, "vocab_size"),
    }


In [ ]:
base = dict(hidden_size=64, num_hidden_layers=2, num_attention_heads=8,
            num_key_value_heads=2, max_position_embeddings=256, vocab_size=128)
configs = {
    "dense_gqa": dict(base),
    "toy_local_moe": dict(base, num_local_experts=4, num_experts_per_tok=2),
    "toy_mla_moe": dict(base, kv_lora_rank=16, n_routed_experts=8,
                        num_experts_per_tok=2, n_shared_experts=1),
}
for name, cfg in configs.items():
    print(name, passport(cfg))

assert "MoE" in passport(configs["toy_local_moe"])["ffn"]
assert passport(configs["toy_mla_moe"])["attention"].startswith("MLA")
for change in ({"num_key_value_heads": 3}, {"num_attention_heads": True},
               {"num_local_experts": 4, "num_experts_per_tok": 5},
               {"num_local_experts": 4, "num_experts": 8, "num_experts_per_tok": 2}):
    try:
        passport(dict(base, **change))
    except ValueError:
        pass
    else:
        raise AssertionError("invalid configuration accepted")


Теперь запишем системный паспорт. Поля `model`, `latency` и `cost` особенно важны:
мы используем подготовленные ответы, поэтому скорость этой проверки нельзя выдать за
скорость языковой модели. Затем воспроизведём вопрос, документы и три случая из §12.

In [ ]:
system_passport = {'task': 'Ответить о сроке хранения журнала с подтверждением', 'documents': ['A', 'B'], 'model': 'Учебный заменитель: заранее подготовленные ответы', 'permissions': 'Только чтение; сеть и запись запрещены', 'memory': 'Отдельно для каждого вопроса', 'acceptance': 'Верный срок и заранее размеченная подтверждающая цитата', 'latency': 'Не измерена для LLM', 'cost': 'Не измерена для LLM'}
print(json.dumps(system_passport, ensure_ascii=False, indent=2))

### Один вопрос: результат проверки

Ассистент по документам помогает ответить на вопрос и показать, откуда взяты сведения.
Начнём с двух вымышленных инструкций из отдельного набора `document-assistant-v1`.
Одна устанавливает срок хранения журнала эксперимента, другая — временной выгрузки.
Тексты встроены ниже: редактирование лекции не меняет данные опыта.

В первой ячейке прочитайте вопрос и оба документа. Затем рассмотрим три **подготовленных**
ответа. Нейросеть и поиск здесь не запускаются: мы исследуем правило приёмки результата.
Поле `days` хранит целое число дней, `document_id` — имя источника, `quote` — цитату.
Эта структурированная форма позволяет увидеть проверку без задачи понимания свободного текста.

In [ ]:
DOCUMENTS = {'A': 'Журнал эксперимента хранится 30 дней после завершения эксперимента.', 'B': 'Временная выгрузка удаляется через 7 дней после создания.'}
QUESTION = 'Сколько дней нужно хранить журнал эксперимента после его завершения?'
print(QUESTION)
for name, passage in DOCUMENTS.items():
    print(name, passage)


У ответа могут быть разные дефекты. Срок 7 дней неверен для журнала, даже если
ссылка ведёт к существующему документу A. Срок 30 дней верен, но документ B его не
подтверждает. Поэтому функция возвращает несколько признаков: правильность числа,
наличие цитаты в документе и соответствие заранее размеченному подтверждению.

Поле `required_quote` содержит полный эталонный фрагмент для этого вопроса. Проверка
ниже намеренно узкая: пересказ другими словами она может отклонить. Её нельзя применять
как универсальную проверку смысла ответа.

In [ ]:
def check_prepared_answer(answer, documents, reference):
    """Проверить срок и точную эталонную цитату для одного фиксированного вопроса."""
    if type(answer.get("days")) is not int or answer["days"] < 0:
        raise ValueError("days должно быть неотрицательным целым")
    doc_id = answer.get("document_id")
    quote = answer.get("quote")
    if not isinstance(doc_id, str) or not isinstance(quote, str):
        raise ValueError("document_id и quote должны быть строками")
    answer_correct = answer["days"] == reference["days"]
    quote_exists = bool(quote) and doc_id in documents and quote in documents[doc_id]
    # Подтверждающий фрагмент заранее размечен человеком для этого вопроса.
    supported = (quote_exists and answer_correct
                 and doc_id == reference["document_id"]
                 and reference["required_quote"] in quote)
    return dict(answer_correct=answer_correct, quote_exists=quote_exists,
                supported=supported, accepted=answer_correct and supported)

In [ ]:
REFERENCE_ANSWER = {'days': 30, 'document_id': 'A', 'required_quote': 'Журнал эксперимента хранится 30 дней после завершения эксперимента.'}
prepared = [{'days': 30, 'document_id': 'A'}, {'days': 7, 'document_id': 'A'}, {'days': 30, 'document_id': 'B'}]
answers = [dict(row, quote=DOCUMENTS[row["document_id"]]) for row in prepared]
checks = [check_prepared_answer(row, DOCUMENTS, REFERENCE_ANSWER) for row in answers]
labels = ['Верный ответ и источник', 'Неверный срок', 'Источник о другом объекте']
for label, result in zip(labels, checks):
    print(label, result)
assert checks == [{'answer_correct': True, 'quote_exists': True, 'supported': True, 'accepted': True}, {'answer_correct': False, 'quote_exists': True, 'supported': False, 'accepted': False}, {'answer_correct': True, 'quote_exists': True, 'supported': False, 'accepted': False}]


Принимается только первый ответ. Второй ошибается в сроке, третий — в подтверждении.
Попробуйте заменить цитату первого ответа одной буквой или выдуманным предложением:
проверка должна перестать принимать его. В М17 мы построим поиск и проследим происхождение
фрагментов, а в М19 рассмотрим оценку произвольных результатов. Здесь мы установили
смысл одной принятой задачи — знаменателя будущего расчёта стоимости.

## Демо 2. Что изменилось между наблюдениями?

Второй проект учит обновлять паспорт по источникам. Здесь реестр вымышленный, чтобы
опыт не зависел от новых выпусков. В `previous` и `current` дата обозначает наблюдение.
Изменение только даты не является новым свойством модели. Сравниваем поля по именам,
сохраняем неизвестный входной контекст как `None`, отдельно учитываем предел выхода.
В настоящем реестре к каждому изменению нужна ссылка на первоисточник.

In [ ]:
previous = {
    "model-a": {"access": "API", "context": 128_000, "max_output": 16_000, "date": "2026-08-06"},
    "model-b": {"access": "promised_weights", "context": 64_000, "max_output": 8_000, "date": "2026-08-06"},
    "model-d": {"access": "API", "context": 64_000, "max_output": 8_000, "date": "2026-08-06"},
}
current = {
    "model-a": {"access": "API", "context": 128_000, "max_output": 32_000, "date": "2026-10-11"},
    "model-b": {"access": "published_weights", "context": 64_000, "max_output": 8_000, "date": "2026-10-11"},
    "model-c": {"access": "restricted", "context": None, "max_output": 1_000_000, "date": "2026-10-11"},
    "model-d": {"access": "API", "context": 64_000, "max_output": 8_000, "date": "2026-10-11"},
}

def registry_changes(previous, current):
    changes = []
    for name in sorted(previous.keys() | current.keys()):
        if name not in previous:
            changes.append((name, "added", current[name]))
        elif name not in current:
            changes.append((name, "removed", previous[name]))
        else:
            delta = {key: (previous[name].get(key), current[name].get(key))
                     for key in sorted(previous[name].keys() | current[name].keys())
                     if key != "date" and previous[name].get(key) != current[name].get(key)}
            if delta:
                changes.append((name, "changed", delta))
    return changes

changes = registry_changes(previous, current)
for change in changes:
    print(change)
assert {name for name, *_ in changes} == {"model-a", "model-b", "model-c"}
assert current["model-c"]["context"] is None
assert registry_changes({"x":{"date":"old"}}, {"x":{"date":"new"}}) == []
assert registry_changes({"x":{}}, {}) == [("x", "removed", {})]


## Переход к полному проекту

Сначала выберите один доступный артефакт и заполните два паспорта. У структурных
свойств запишите первоисточник и точную ревизию, у измерений — данные, настройки и
правило приёмки. Неизвестные поля оставьте неизвестными. Затем можно расширить сравнение
до пяти моделей и повторить наблюдение через несколько недель. Дату наблюдения храните
отдельно от даты выпуска.

Для реального ассистента потребуется выполнить поиск и генерацию, сохранить полученные
ответы, а затем оценить их на отложенных вопросах. В М17 и М19 появятся необходимые
механизмы. До такого запуска этот опыт подтверждает только работу классификатора,
правил узкой проверки и сравнения записей; он не измеряет качество какой-либо LLM.